---
# FIGER Conversion
> Thorsten Trinkaus
---

This Notebook was created to convert the FIGER data (train.data.gz) from a
compressed stream of Protocol Buffer mention records into a form that is usable
for this project (JSONL). See compile.sh for how we got entity_pb2.py from
entity.proto.

---
## Imports
---

In [1]:
import entity_pb2
from datasets import Dataset, DatasetDict

---
We first analyze the .proto file descriptor to understand its structure.

In [2]:
print("top-level messages:", list(entity_pb2.DESCRIPTOR.message_types_by_name.keys()))
print("top-level enums:", list(entity_pb2.DESCRIPTOR.enum_types_by_name.keys()))

top-level messages: ['Mention']
top-level enums: []


---
Now we can convert the data into a single dataset.

In [3]:
# Generator that streams FIGER protobuf records.
def gen():
    import gzip, importlib
    from google.protobuf.json_format import MessageToDict

    # Read a protobuf varint length prefix from a byte stream.
    # FIGER's train.data.gz stores messages as length delimited protobuf
    # records:
    # [varint mes_size][mes_bytes][varint mes_size][mes_bytes]...
    def read_varint(stream):
        shift = 0
        result = 0
        while True:
            b = stream.read(1)
            if not b:
                # End of File
                return None
            byte = b[0]
            result |= (byte & 0x7F) << shift
            if not (byte & 0x80):
                return result
            shift += 7

    # Import the protobuf module.
    pb2 = importlib.import_module("entity_pb2")
    
    # Get the message class for Mention as defined in entity.proto.
    message_cls = getattr(pb2, "Mention")

    # Open the compressed data file and stream-decode messages one by one.
    with gzip.open("train.data.gz", "rb") as f:
        while True:

            # Message length
            size = read_varint(f)
            if size is None:
                # End of File
                break

            # Read and parse message.
            blob = f.read(size)
            msg = message_cls()
            msg.ParseFromString(blob)

            # Convert the message to a python dict.
            # This structure is JSON friendly.
            # always_print_fields_with_no_presence includes default-valued
            # fields as well.
            dict_ = MessageToDict(
                msg,
                always_print_fields_with_no_presence=True,
            )

            # Yield dicts one by one for Hugging Face Datasets.
            yield dict_

# Build the dataset from the generator.
ds = Dataset.from_generator(gen)

Generating train split: 0 examples [00:00, ? examples/s]

Sanity Check:

In [4]:
print(ds.column_names)
print(ds[0])
print("Size of dataset:", len(ds))

['start', 'end', 'tokens', 'posTags', 'deps', 'entityName', 'labels', 'sentid', 'fileid', 'features']
{'start': 11, 'end': 13, 'tokens': ['The', 'band', 'also', 'shared', 'membership', 'with', 'the', 'similar', ',', 'defunct', 'group', 'Out', 'Hud', '(', 'including', 'Tyler', 'Pope', ',', 'who', 'has', 'played', 'with', 'LCD', 'Soundsystem', 'and', 'written', 'music', 'for', 'Cake', ')', '.'], 'posTags': ['DT', 'NN', 'RB', 'VBD', 'NN', 'IN', 'DT', 'JJ', ',', 'JJ', 'NN', 'IN', 'NNP', '-LRB-', 'VBG', 'NNP', 'NNP', ',', 'WP', 'VBZ', 'VBN', 'IN', 'NNP', 'NNP', 'CC', 'VBN', 'NN', 'IN', 'NNP', '-RRB-', '.'], 'deps': [{'dep': 0, 'gov': 1, 'type': 'det'}, {'dep': 1, 'gov': 3, 'type': 'nsubj'}, {'dep': 2, 'gov': 3, 'type': 'advmod'}, {'dep': 4, 'gov': 3, 'type': 'dobj'}, {'dep': 6, 'gov': 10, 'type': 'det'}, {'dep': 7, 'gov': 10, 'type': 'amod'}, {'dep': 9, 'gov': 10, 'type': 'amod'}, {'dep': 10, 'gov': 4, 'type': 'prep_with'}, {'dep': 11, 'gov': 4, 'type': 'amod'}, {'dep': 12, 'gov': 4, 'type'

---
We still need to store the new dataset, so the previous steps do not need to be repeated each time.

In [5]:
ds.to_json("figer.jsonl")

Creating json from Arrow format:   0%|          | 0/4048 [00:00<?, ?ba/s]

5882708964

---
For this project we require train/dev/test splits. We will use a 80/10/10 split.

In [6]:
# Test split (fixed seed for reproducibility):
split_1 = ds.train_test_split(test_size=0.1, seed=42)

# Dev split (0.1111 * 0.9 = 0.1):
split_2 = split_1["train"].train_test_split(test_size=0.1111, seed=42)

splits = DatasetDict(
    {
        "train": split_2["train"],
        "dev": split_2["test"],
        "test": split_1["test"]
    }
)

print(splits)

DatasetDict({
    train: Dataset({
        features: ['start', 'end', 'tokens', 'posTags', 'deps', 'entityName', 'labels', 'sentid', 'fileid', 'features'],
        num_rows: 3237703
    })
    dev: Dataset({
        features: ['start', 'end', 'tokens', 'posTags', 'deps', 'entityName', 'labels', 'sentid', 'fileid', 'features'],
        num_rows: 404668
    })
    test: Dataset({
        features: ['start', 'end', 'tokens', 'posTags', 'deps', 'entityName', 'labels', 'sentid', 'fileid', 'features'],
        num_rows: 404708
    })
})


In [7]:
# Store each split:
splits["train"].to_json("figer_train.jsonl")
splits["dev"].to_json("figer_dev.jsonl")
splits["test"].to_json("figer_test.jsonl")

Creating json from Arrow format:   0%|          | 0/3238 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/405 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/405 [00:00<?, ?ba/s]

588741645